# Epoch Spring Camp 2026 - Take Home Task 3

In this task, you will build and compare two recommender system models:

- **Matrix Factorization (MF)** using a dot product of embeddings  
- **Neural Collaborative Filtering (MLP-based)** using a multi-layer perceptron  

You are provided with:
- Preprocessed interaction data
- Evaluation pipeline

You are expected to implement:
- Negative sampling
- Model architectures
- Training loop

---

The purpose of this task is to understand how neural networks can model user–item interactions beyond simple similarity.

## Imports

In [37]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import random
import pandas as pd
import numpy as np

## Loading Data

We begin by loading the interaction data.

In [38]:
df = pd.read_csv("interactions.csv")  # columns: user_id, movie_id
items=df['item_id'].unique()
users=df['user_id'].unique()

print("Num users:", df['user_id'].nunique())
print("Num items:", df['item_id'].nunique())
print("Num interactions:", len(df))

Num users: 942
Num items: 1447
Num interactions: 55375


Each row represents a **positive interaction** (implicit feedback).

## Train / Test Split

We split the data into training and testing sets.

The model will learn from the training data and be evaluated on unseen interactions in the test set.

In [39]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

## PyTorch Dataset

We now wrap our processed data into a PyTorch `Dataset`.

This allows us to:
- Access individual samples as `(user, item, label)`
- Easily plug into a `DataLoader` for batching

You do not need to modify this part.

In [40]:

class InteractionDataset(Dataset):
    def __init__(self, df, all_items, num_neg=4):
        """
        Pass ONLY the positive interactions (train_df) to this dataset.
        Do not pass pre-sampled negatives.
        """
        self.users = df['user_id'].values.astype(int)
        self.pos_items = df['item_id'].values.astype(int)
        self.all_items = all_items.astype(int)
        self.num_neg = num_neg

        # Precompute a hash map of user -> set(interacted_items) for O(1) lookup during training
        self.user_interacted = df.groupby('user_id')['item_id'].apply(set).to_dict()

    def __len__(self):
        # The virtual size of the dataset is expanded by the negative ratio
        return len(self.users) * (1 + self.num_neg)

    def __getitem__(self, idx):
        # Determine the base interaction index and whether this call should return a positive or negative sample
        pos_idx = idx // (1 + self.num_neg)
        is_pos = (idx % (1 + self.num_neg)) == 0

        u = self.users[pos_idx]

        if is_pos:
            i = self.pos_items[pos_idx]
            label = 1.0
        else:
            # Dynamically sample a negative item on the fly
            i = random.choice(self.all_items)
            interacted = self.user_interacted.get(u, set())

            # Reroll if the randomly selected item is actually a positive interaction
            while i in interacted:
                i = random.choice(self.all_items)
            label = 0.0

        return torch.tensor(u, dtype=torch.long), torch.tensor(i, dtype=torch.long), torch.tensor(label, dtype=torch.float32)

## DataLoader

The `DataLoader` will:
- Batch the data
- Shuffle the training data
- Feed it to the model during training

In [41]:
train_data = train_df

train_ds,valid_ds=train_test_split(train_data,test_size=0.1,random_state=42)
train_loader = DataLoader(InteractionDataset(train_ds,all_items=items,num_neg=4), batch_size=256, shuffle=True)
valid_loader = DataLoader(InteractionDataset(valid_ds,all_items=items,num_neg=4), batch_size=256, shuffle=False)

Why are we sampling negatives only for the training data?

## Quick Exploration

Before building models, take a moment to explore the data.

Try to understand:
- How many interactions each user has
- How popular certain items are

This can give intuition about the dataset.

In [42]:
# Interactions per user
user_counts = df['user_id'].value_counts()
print(user_counts.describe())

# Interactions per item
item_counts = df['item_id'].value_counts()
print(item_counts.describe())

count    942.000000
mean      58.784501
std       54.696664
min        3.000000
25%       19.000000
50%       39.500000
75%       80.750000
max      378.000000
Name: count, dtype: float64
count    1447.000000
mean       38.268832
std        57.956847
min         1.000000
25%         3.000000
50%        13.000000
75%        47.500000
max       501.000000
Name: count, dtype: float64


## Baseline Model: Matrix Factorization (MF)

We represent:
- Each **user** as a vector (embedding)
- Each **item** as a vector (embedding)

To predict interaction:
- Compute the **dot product** between user and item embeddings

This gives a **score** indicating how likely the user is to interact with the item.

---

### Your Task

Implement a model that:
1. Learns user and item embeddings
2. Computes their dot product as the output score

In [43]:
class MF(nn.Module):
    def __init__(self, num_users, num_items, emb_dim):
        super().__init__()

        # TODO:
        # - Define user embedding layer
        # - Define item embedding layer
        self.user_emb = nn.Embedding(num_users, emb_dim)
        self.item_emb = nn.Embedding(num_items, emb_dim)

    def forward(self, user, item):
        # TODO:
        # - Get user and item embeddings
        # - Compute dot product
        # - Return a single score per pair
        user_emb = self.user_emb(user)
        item_emb = self.item_emb(item)
        return (user_emb * item_emb).sum(1)

## Training the MF Model

Now train your Matrix Factorization model.

You will need to:
- Define a loss function
- Define an optimizer
- Iterate over the DataLoader
- Update model parameters

---

### Hints

- Use **Binary Cross Entropy loss**
- Apply **sigmoid** to model outputs if needed
- Typical steps:
  - forward pass
  - compute loss
  - backward pass
  - optimizer step

In [44]:
def train(model, dataloader, val_dataloader, epochs, patience=100):
    if torch.cuda.is_available():
        device = torch.device('cuda')
    else:
        device = torch.device('cpu')

    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.BCEWithLogitsLoss()
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

    best_val_loss = float('inf')
    early_stop_counter = 0
    losses = []

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for user, item, label in dataloader:
            optimizer.zero_grad()
            relevance_scores = model(user.to(device), item.to(device))
            label = label.to(device).float()

            loss = criterion(relevance_scores, label)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_train_loss = total_loss / len(dataloader)
        losses.append(avg_train_loss)

        # Validation
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for user, item, label in val_dataloader:
                relevance_scores = model(user.to(device), item.to(device))
                label = label.to(device).float()
                loss = criterion(relevance_scores, label)
                val_loss += loss.item()

        avg_val_loss = val_loss / len(val_dataloader)
        print(f"Epoch {epoch+1}: Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}")

        # Scheduler step
        scheduler.step(avg_val_loss)

        # Early Stopping logic
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            early_stop_counter = 0
            # Optional: Save best model
            best_model_state = model.state_dict()
        else:
            early_stop_counter += 1
            if early_stop_counter >= patience:
                print("Early stopping triggered.")
                model.load_state_dict(best_model_state)
                break

    return losses

In [45]:
# TODO:
# 1. Initialize MF model
MF_model = MF(num_users=df['user_id'].nunique(), num_items=df['item_id'].nunique(), emb_dim=16)
# 2. Train itee
train(MF_model, train_loader,valid_loader, epochs=100)

Epoch 1: Train Loss: 1.6400, Val Loss: 1.5564
Epoch 2: Train Loss: 1.4860, Val Loss: 1.4335
Epoch 3: Train Loss: 1.3437, Val Loss: 1.3116
Epoch 4: Train Loss: 1.2264, Val Loss: 1.2024
Epoch 5: Train Loss: 1.1245, Val Loss: 1.1030
Epoch 6: Train Loss: 1.0348, Val Loss: 1.0227
Epoch 7: Train Loss: 0.9537, Val Loss: 0.9492
Epoch 8: Train Loss: 0.8793, Val Loss: 0.8678
Epoch 9: Train Loss: 0.8044, Val Loss: 0.7987
Epoch 10: Train Loss: 0.7169, Val Loss: 0.7090
Epoch 11: Train Loss: 0.6305, Val Loss: 0.6304
Epoch 12: Train Loss: 0.5461, Val Loss: 0.5513
Epoch 13: Train Loss: 0.4753, Val Loss: 0.4995
Epoch 14: Train Loss: 0.4300, Val Loss: 0.4691
Epoch 15: Train Loss: 0.4006, Val Loss: 0.4445
Epoch 16: Train Loss: 0.3812, Val Loss: 0.4288
Epoch 17: Train Loss: 0.3678, Val Loss: 0.4207
Epoch 18: Train Loss: 0.3571, Val Loss: 0.4176
Epoch 19: Train Loss: 0.3487, Val Loss: 0.4125
Epoch 20: Train Loss: 0.3438, Val Loss: 0.4054
Epoch 21: Train Loss: 0.3374, Val Loss: 0.4098
Epoch 22: Train Loss: 

[1.640049061665027,
 1.485975908376133,
 1.3437479911933967,
 1.2264361802359447,
 1.1244901067485247,
 1.034805192873934,
 0.9537042435234716,
 0.8793148971491509,
 0.8044329789850923,
 0.7168667632440854,
 0.6305305299771153,
 0.5460549089125093,
 0.47527938197451775,
 0.43003399668211195,
 0.40057660250975935,
 0.3812048510960345,
 0.36782653302681156,
 0.35708771655587085,
 0.34866106253839424,
 0.34380136187330596,
 0.33741157212138023,
 0.33150660955324407,
 0.326260525584986,
 0.3211410731957109,
 0.31535524824234884,
 0.312012855790852,
 0.30603175507965996,
 0.30125451005698783,
 0.2963295029010026,
 0.29204561508742466,
 0.2883622929817293,
 0.2831924706123607,
 0.28014967834459803,
 0.2764785727426238,
 0.2729305577905555,
 0.2696668840610017,
 0.26613253175217627,
 0.2624022687889339,
 0.261648546689129,
 0.25899319010230787,
 0.25681994606732406,
 0.25781067786917605,
 0.25588832655981353,
 0.25526128801133424,
 0.25682082875549256,
 0.2559658021729473,
 0.2560449340912741

## Evaluation (Hit@K)

We evaluate the model using a ranking-based metric.

For each user:
- Take one positive item
- Sample multiple negative items
- Rank all items using the model
- Check if the positive item is in the top-K

This is called **Hit@K**.

In [46]:
def hit_at_k(model, test_df, full_df, K=10, num_neg=100):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.eval()
    model.to(device)

    hits = 0
    total = len(df) # We evaluate every positive interaction in the test set

    # Map interactions for quick lookup
    interacted_items = full_df.groupby('user_id')['item_id'].apply(set).to_dict()
    all_items = full_df['item_id'].unique()

    with torch.no_grad(): # Disable gradient calculation for speed/memory
        for _, row in df.iterrows():
            u = int(row['user_id'])
            pos_item = int(row['item_id'])

            # 1. Sample Negatives
            negatives = []
            while len(negatives) < num_neg:
                neg_item = np.random.choice(all_items)
                if neg_item not in interacted_items.get(u, set()):
                    negatives.append(neg_item)

            # 2. Prepare Tensors
            # We need a list of the 1 positive + 100 negatives
            item_list = [pos_item] + negatives
            user_tensor = torch.tensor([u] * (num_neg + 1)).to(device)
            item_tensor = torch.tensor(item_list).to(device)

            # 3. Get Scores
            scores = model(user_tensor, item_tensor)

            # 4. Rank and Check Hit
            # We want to see if the item at index 0 (the positive) is in the top K
            # torch.topk returns values and indices of the highest scores
            _, top_indices = torch.topk(scores, K)

            top_indices = top_indices.cpu().numpy()
            if 0 in top_indices:
                hits += 1

    return hits / total


In [47]:


# 3. Evaluate it
hit_rate = hit_at_k(MF_model, test_df, df, K=10, num_neg=100)
print(f"Hit@10: {hit_rate:.4f}")

Hit@10: 0.7513


If your score is low, try playing with the hyperparameters before moving on! Try sampling more negatives, or playing with the embedding dimensions.

Conversely, if your score is high, play with the values of K or number of negatives in evaluation (dec K, inc negatives).

## Neural Model: MLP-Based Recommender

Instead of using a dot product, we can learn a more flexible interaction function using a neural network.

Approach:
- Learn user and item embeddings
- **Concatenate** them
- Pass through a **Multi-Layer Perceptron (MLP)**

This allows the model to capture more complex relationships than simple similarity.

---

### Your Task

Implement a model that:
1. Learns user and item embeddings
2. Concatenates them
3. Passes them through an MLP to produce a score

The choice of activation functions is upto you.

In [48]:
class MLP(nn.Module):
    def __init__(self, num_users, num_items, emb_dim):
        super().__init__()

        # TODO:
        # - Define user embedding layer
        # - Define item embedding layer
        # - Define MLP layers
        self.user_emb=nn.Embedding(num_users, emb_dim)
        self.item_emb=nn.Embedding(num_items, emb_dim)
        self.mlp = nn.Sequential(
            nn.Linear(2 * emb_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, user, item):
        # TODO:
        # - Get embeddings
        # - Concatenate user and item embeddings
        # - Pass through MLP
        # - Return a single score
        user_emb = self.user_emb(user)
        item_emb = self.item_emb(item)
        concatenated = torch.cat([user_emb, item_emb], dim=1)
        return self.mlp(concatenated)[:,0]

## Train and Evaluate the MLP Model

Repeat the same steps as before:
- Train the model
- Evaluate on the test set

Compare its performance with the MF model.

In [49]:
# TODO:
# 1. Initialize MLP model
# 2. Train it
# 3. Evaluate it
MLP_model = MLP(num_users=df['user_id'].nunique(), num_items=df['item_id'].nunique(), emb_dim=16)
train(MLP_model, train_loader,valid_loader, epochs=300)
hit_rate = hit_at_k(MLP_model, test_df, df, K=10, num_neg=100)
print(f"Hit@10: {hit_rate:.4f}")


Epoch 1: Train Loss: 0.4721, Val Loss: 0.4189
Epoch 2: Train Loss: 0.3852, Val Loss: 0.3897
Epoch 3: Train Loss: 0.3675, Val Loss: 0.3873
Epoch 4: Train Loss: 0.3632, Val Loss: 0.3881
Epoch 5: Train Loss: 0.3616, Val Loss: 0.3839
Epoch 6: Train Loss: 0.3607, Val Loss: 0.3871
Epoch 7: Train Loss: 0.3584, Val Loss: 0.3869
Epoch 8: Train Loss: 0.3576, Val Loss: 0.3848
Epoch 9: Train Loss: 0.3539, Val Loss: 0.3866
Epoch 10: Train Loss: 0.3522, Val Loss: 0.3909
Epoch 11: Train Loss: 0.3523, Val Loss: 0.3849
Epoch 12: Train Loss: 0.3493, Val Loss: 0.3894
Epoch 13: Train Loss: 0.3498, Val Loss: 0.3906
Epoch 14: Train Loss: 0.3484, Val Loss: 0.3885
Epoch 15: Train Loss: 0.3470, Val Loss: 0.3871
Epoch 16: Train Loss: 0.3471, Val Loss: 0.3914
Epoch 17: Train Loss: 0.3463, Val Loss: 0.3910
Epoch 18: Train Loss: 0.3466, Val Loss: 0.3898
Epoch 19: Train Loss: 0.3471, Val Loss: 0.3911
Epoch 20: Train Loss: 0.3472, Val Loss: 0.3892
Epoch 21: Train Loss: 0.3471, Val Loss: 0.3897
Epoch 22: Train Loss: 

## Comparison & Analysis

You have now implemented:
- Matrix Factorization (MF)
- MLP-based recommender

---

### Compare the Models

- What score did each model achieve?
- Which model performed better?

---

### Think & Reflect

- Why might the MLP model outperform MF?
- In what cases might MF perform just as well or better?
- How does embedding size affect performance?
- Did you observe any signs of overfitting?

---

### Some Experiments

If you want to explore further:
- Try different embedding dimensions
- Change number of MLP layers
- Try different activation functions

---

## Bonus Task: Neural Collaborative Filtering (NCF)

In this task, you implemented:
- Matrix Factorization (MF)
- MLP-based recommender

The paper [Neural Collaborative Filtering](https://arxiv.org/pdf/1708.05031) proposes combining both ideas.

---

### Idea

- MF captures **linear interactions**
- MLP captures **nonlinear interactions**

NCF combines both by:
1. Computing MF output
2. Computing MLP output
3. Combining them into a final prediction

---

### Your Task

Design a model that:
- Uses both MF and MLP components
- Combines their outputs
- Trains end-to-end

---

### Hints

- You can:
  - Concatenate MF and MLP representations
  - Or combine their final scores
- Think about:
  - Should embeddings be shared or separate?
  - How to balance both components?

---

Does combining both approaches improve performance over MF and MLP individually?

In [50]:
class NCF(nn.Module):
    def __init__(self, num_users, num_items, mf_dim=16, mlp_dim=16, layers=[64, 32, 16]):
        super(NCF, self).__init__()

        # Matrix Factorization (GMF) component
        self.mf_user_emb = nn.Embedding(num_users, mf_dim)
        self.mf_item_emb = nn.Embedding(num_items, mf_dim)

        # MLP component
        self.mlp_user_emb = nn.Embedding(num_users, mlp_dim)
        self.mlp_item_emb = nn.Embedding(num_items, mlp_dim)

        mlp_modules = []
        input_size = 2 * mlp_dim
        for output_size in layers:
            mlp_modules.append(nn.Linear(input_size, output_size))
            mlp_modules.append(nn.ReLU())
            input_size = output_size
        self.mlp_network = nn.Sequential(*mlp_modules)

        # Final Prediction Layer (Combines both outputs)
        self.prediction_layer = nn.Linear(mf_dim + layers[-1], 1)

    def forward(self, user, item):
        # GMF Branch
        mf_user = self.mf_user_emb(user)
        mf_item = self.mf_item_emb(item)
        mf_output = mf_user * mf_item # element-wise product

        # MLP Branch
        mlp_user = self.mlp_user_emb(user)
        mlp_item = self.mlp_item_emb(item)
        mlp_input = torch.cat([mlp_user, mlp_item], dim=1)
        mlp_output = self.mlp_network(mlp_input)

        # Concatenate GMF and MLP outputs
        combined = torch.cat([mf_output, mlp_output], dim=1)

        # Output layer
        return self.prediction_layer(combined).squeeze()

### Hyperparameter Tuning & Regularization

Let's try to improve the score by:
- Increasing the embedding dimension to 32.
- Adding `weight_decay` to the optimizer to regularize the embeddings.
- Tuning the MLP architecture.

In [ ]:
# Re-initialize NCF with larger embeddings
improved_ncf = NCF(
    num_users=df['user_id'].nunique(),
    num_items=df['item_id'].nunique(),
    mf_dim=32,
    mlp_dim=32,
    layers=[128, 64, 32]
)

# Updated training call with Weight Decay
def train_improved(model, dataloader, val_dataloader, epochs=100, lr=0.001, weight_decay=1e-5):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.BCEWithLogitsLoss()

    # Re-using your training logic with the added weight decay
    return train(model, dataloader, val_dataloader, epochs=epochs)

print("Training improved NCF model...")
train_improved(improved_ncf, train_loader, valid_loader)

# Final Evaluation
final_hit_rate = hit_at_k(improved_ncf, test_df, df, K=10, num_neg=100)
print(f"Improved NCF Hit@10: {final_hit_rate:.4f}")

Training improved NCF model...
Epoch 1: Train Loss: 0.4485, Val Loss: 0.4032
Epoch 2: Train Loss: 0.3745, Val Loss: 0.3890
Epoch 3: Train Loss: 0.3647, Val Loss: 0.3882
Epoch 4: Train Loss: 0.3585, Val Loss: 0.3910
Epoch 5: Train Loss: 0.3556, Val Loss: 0.3920
Epoch 6: Train Loss: 0.3506, Val Loss: 0.3910
Epoch 7: Train Loss: 0.3429, Val Loss: 0.3893
Epoch 8: Train Loss: 0.3394, Val Loss: 0.3943
Epoch 9: Train Loss: 0.3352, Val Loss: 0.3996
Epoch 10: Train Loss: 0.3289, Val Loss: 0.4010
Epoch 11: Train Loss: 0.3270, Val Loss: 0.4064
Epoch 12: Train Loss: 0.3241, Val Loss: 0.4028
Epoch 13: Train Loss: 0.3205, Val Loss: 0.4068
Epoch 14: Train Loss: 0.3174, Val Loss: 0.4131
Epoch 15: Train Loss: 0.3189, Val Loss: 0.4111
Epoch 16: Train Loss: 0.3162, Val Loss: 0.4081
Epoch 17: Train Loss: 0.3150, Val Loss: 0.4131
Epoch 18: Train Loss: 0.3138, Val Loss: 0.4144
Epoch 19: Train Loss: 0.3129, Val Loss: 0.4145
Epoch 20: Train Loss: 0.3126, Val Loss: 0.4121
Epoch 21: Train Loss: 0.3126, Val Loss